# 51 — Explainable Scoring
**Goal:** Generate human-readable explanations for each resume score.

Ch. 50 produced scores; this chapter makes them **defensible**. `ExplainableScorer` subclasses the Ch. 50 rule set and returns, for every dimension, not just a number but a `reason` string — "Matched 1/3 required skills" — plus the raw `details` behind it. The output is a structured dict, so a front end can render "why 46.8?" without the scorer ever writing prose.

**Why it matters for resumes / ATS:** an unexplained score is a liability. Candidates contest it, recruiters can't act on it, and engineers can't debug it. When every dimension carries a reason, a low score becomes an actionable message ("add a Summary section, learn SQL") instead of a mystery — and a high score becomes evidence you can show a client or a hiring manager.

![ATS Scoring Engine](../../../assets/images/ats_scoring_engine_1785491176998.png)

> **Figure:** The ATS Scoring Engine — showing the rule engine → semantic scorer → explainable output breakdown with weighted component scores.

The figure is the architectural map for this whole block: the **rule engine** from Ch. 50 produces the dimension scores, a **semantic scorer** (added in later chapters) refines them, and an **explainable output** layer attaches reasons. The weighted component scores on the right are exactly the dict `score_with_explanations()` returns — each bar is one `{score, reason}` pair, so what you see in the chart is what you get in the data.

## 1. Score Breakdown with Explanations

`ExplainableScorer` adds one method to the Ch. 50 `ATSScorer`: `score_with_explanations()`, which returns `(total, explanations)` — a dict keyed by dimension, each entry holding `score`, `reason`, and sometimes `details`.

**What the code does:**
- `skill_match` — calls `skill_match_score()`, then converts the percentage back into a count for the reason: `Matched 1/3 required skills`; `details` carries the raw found/required skill lists.
- `experience` — the reason is simply `5 years experience vs 3 required`; the number came from Ch. 50's band table.
- `format` — recomputes the missing-section list so the reason can name it: `Missing sections: ['summary', ...]` or `All sections present`.
- `boolean` — only present when `must_have_terms` is passed; reason reports `Found 1/2 required terms`.
- `total` — weighted sum **over whichever dimensions were computed**; dimensions never measured contribute nothing to the total.

**Expected (verified by running):** with the sample resume the call returns `Total score: 46.8/100` with `skill_match 66.7`, `experience 100.0`, `format 10.0`, `boolean 50.0`. Two things stand out. First, `format` is 10.0: four sections "missing" (the `r"\\b"` word-boundary trap from Ch. 50, confirmed here) plus a 30-point short-text penalty. Second, the total is out of a 70-point budget, not 100 — `education`, `bullet_quality`, and `duration` were never computed, so 30% of the weight silently vanishes. Explaining every number is exactly how you catch that.

In [ ]:
class ExplainableScorer(ATSScorer):
    def score_with_explanations(self, resume_text, jd_text, resume_skills, jd_skills,
                                  resume_years=5, required_years=3, must_have_terms=None):
        explanations = {}
        
        # Skill match
        s = self.skill_match_score(resume_skills, jd_skills)
        explanations["skill_match"] = {
            "score": round(s, 1),
            "reason": f"Matched {int(s/100*len(jd_skills)) if jd_skills else 0}/{len(jd_skills)} required skills",
            "details": {"found": resume_skills, "required": jd_skills}
        }
        
        s2 = self.experience_score(resume_years, required_years)
        explanations["experience"] = {
            "score": round(s2, 1),
            "reason": f"{resume_years} years experience vs {required_years} required"
        }
        
        s3 = self.format_score(resume_text)
        missing = []
        for sec in ["summary", "experience", "education", "skills"]:
            if not re.search(r"\\b" + sec + r"\\b", resume_text, re.IGNORECASE):
                missing.append(sec)
        explanations["format"] = {
            "score": round(s3, 1),
            "reason": f"Missing sections: {missing}" if missing else "All sections present"
        }
        
        if must_have_terms:
            s4 = self.boolean_check(resume_text, must_have_terms)
            explanations["boolean"] = {
                "score": round(s4, 1),
                "reason": f"Found {int(s4/100*len(must_have_terms))}/{len(must_have_terms)} required terms"
            }
        
        total = sum(ex["score"] * self.weights[k] for k, ex in explanations.items())
        return round(total, 1), explanations

es = ExplainableScorer()
score, exps = es.score_with_explanations(
    "Data scientist with Python, NLP, TensorFlow. 5 years. MS Computer Science.",
    "Senior data scientist, Python NLP required, 5+ years",
    ["Python", "NLP", "TensorFlow"], ["Python", "NLP", "SQL"],
    must_have_terms=["Python", "ML"]
)
print(f"Total score: {score}/100\n")
for dim, exp in exps.items():
    print(f"  [{dim:15s}] {exp['score']:5.1f}/100 | {exp['reason']}")

## 2. Visualization of Score Breakdown

A dict of numbers is still hard to scan; the chart makes the breakdown **visual**. Two side-by-side bar charts answer different questions: *which dimensions scored well?* (raw) versus *which dimensions actually moved the total?* (weighted).

**What the code does:** builds three parallel lists — `scores` (raw 0–100), `weights`, and `weighted = score × weight` — then draws `ax1` as raw bars and `ax2` as weighted bars, saves the figure to `/tmp/ats_breakdown.png`, and shows it. The right-hand chart is the honest one: `skill_match` at 66.7 looks mediocre on the left but contributes `66.7 × 0.35 ≈ 23.3` points on the right — nearly half the total — while `format`'s 10.0 contributes only 1.0 point. Score × weight, not score alone, is what ranking should optimize.

**Caution:** `savefig('/tmp/ats_breakdown.png')` is POSIX-flavored. On Windows the absolute path resolves to the current drive's root (e.g. `D:\tmp\ats_breakdown.png`), which usually doesn't exist — verified: the cell renders the charts, then raises `FileNotFoundError` on the save. Create that directory or change the path.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

dims = list(exps.keys())
scores = [exps[d]['score'] for d in dims]
weights = [es.weights.get(d, 0.1) for d in dims]
weighted = [s * w for s, w in zip(scores, weights)]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.bar(dims, scores, color='steelblue')
ax1.set_ylabel('Raw Score (0-100)')
ax1.set_title('Dimension Scores')
ax1.tick_params(axis='x', rotation=45)

ax2.bar(dims, weighted, color='coral')
ax2.set_ylabel('Weighted Contribution')
ax2.set_title('Final Contribution (score x weight)')
ax2.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig('/tmp/ats_breakdown.png', dpi=100)
plt.show()
print("\nChart saved to /tmp/ats_breakdown.png")

## Summary: Every score has an explanation. Transparency builds trust with recruiters and candidates.

**A score without a reason is noise — attach a `reason` to every dimension and the system becomes auditable.**

`score_with_explanations()` returns structured `{score, reason, details}` entries, so the 46.8 total decomposes into four explainable parts, each traceable to a Ch. 50 rule. That auditability doubles as a debugging tool: the missing `education`/`bullet_quality`/`duration` weights are visible precisely because the explanation dict only contains what was computed.

This chapter feeds Ch. 52, where the same explainable scorer is embedded in a full end-to-end ATS simulation — parse, extract, score, report.